In [1]:
import sys
import os 
from tqdm import tqdm
from rasterio.enums import Resampling
import glob

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)


Project root added: c:\Users\debas\OneDrive\Desktop\Final Year Project\Main_prj


In [2]:

#utilites

from utils.masking import apply_scl_mask
from utils.stacking import stack_bands, save_scene
from utils.resampling import resample_band, read_band, save_band
from utils.normalize import  normalize
from utils.patching import extract_and_save_patches_streaming


In [3]:
# data path
DRIVE_DIR = r"G:\My Drive\Satelite Project Dataset\Soil_Classification"
DATASET_ROOT = "../Dataset/soil_classification"
LOCAL_PATCH_DIR = os.path.join(DATASET_ROOT, "patches")


os.makedirs(LOCAL_PATCH_DIR, exist_ok=True)
print("Patches will be saved in:", LOCAL_PATCH_DIR)

Patches will be saved in: ../Dataset/soil_classification\patches


In [4]:
def find_band(scene_path, band_key):
    """
    band_key examples:
    'B02_10m', 'B03_10m', 'B04_10m', 'B08_10m',
    'B11_20m', 'SCL_20m'
    """
    files = glob.glob(os.path.join(scene_path, f"*_{band_key}.jp2"))
    
    if len(files) == 0:
        raise FileNotFoundError(f"No file found for {band_key} in {scene_path}")
    
    return files[0]  # Sentinel-2 gives exactly one match

In [29]:
def proccess_scene(scene_name):
    scene_path = os.path.join(DRIVE_DIR, scene_name)
    print(f"\nProcessing scene: {scene_name}")

    # find band files
    b02_path = find_band(scene_path, 'B02_10m')
    b03_path = find_band(scene_path, "B03_10m")
    b04_path = find_band(scene_path, "B04_10m")
    b08_path = find_band(scene_path, "B08_10m")
    b11_path = find_band(scene_path, "B11_20m")
    scl_path = find_band(scene_path, "SCL_20m")

    # read 10m bands
    b02, ref_profile = read_band(b02_path)
    b03, _ = read_band(b03_path)
    b04, _ = read_band(b04_path)
    b08, _ = read_band(b08_path)

    # read 20m bands
    b11_20, b11_profile = read_band(b11_path)
    scl_20, scl_profile = read_band(scl_path)

    # resample to 10m
    b11_10 = resample_band(b11_20, b11_profile, ref_profile, Resampling.bilinear)
    scl_10 = resample_band(scl_20, scl_profile, ref_profile, Resampling.nearest)

    # apply SCL mask
    b02 = apply_scl_mask(b02, scl_10)
    b03 = apply_scl_mask(b03, scl_10)
    b04 = apply_scl_mask(b04, scl_10)
    b08 = apply_scl_mask(b08, scl_10)

    # STREAMING PATCH EXTRACTION (KEY LINE)
    scene_patch_dir = os.path.join(LOCAL_PATCH_DIR, scene_name)

    extract_and_save_patches_streaming(
        b04=b04,
        b03=b03,
        b02=b02,
        b08=b08,
        scl=scl_10,
        patch_size=64,
        stride=32,
        out_dir=scene_patch_dir,
        scene_id=scene_name
    )

    # free memory explicitly
    del b02, b03, b04, b08, scl_10, b11_20





In [30]:
test_scene = os.listdir(DRIVE_DIR)[0]
proccess_scene(test_scene)



Processing scene: scene_1
Saved 34015 patches for scene_1


In [31]:
import numpy as np

scene_patch_dir = os.path.join(LOCAL_PATCH_DIR, test_scene)
files = os.listdir(scene_patch_dir)

print("Number of patches:", len(files))

sample = np.load(os.path.join(scene_patch_dir, files[0]))
print("Sample shape:", sample.shape)
print("Value range:", sample.min(), sample.max())

Number of patches: 42525
Sample shape: (64, 64, 4)
Value range: 0.0 5.444e+03


## create patches for all scenes

In [32]:
scene_names = [
    s for s in os.listdir(DRIVE_DIR)
    if os.path.isdir(os.path.join(DRIVE_DIR, s))
]

print("Total scenes found:", len(scene_names))

for scene_name in tqdm(scene_names):
    proccess_scene(scene_name)


Total scenes found: 11


  0%|          | 0/11 [00:00<?, ?it/s]


Processing scene: scene_1
Saved 34015 patches for scene_1


  9%|▉         | 1/11 [03:02<30:27, 182.79s/it]


Processing scene: scene_2


...

Saved 90416 patches for scene_8

Processing scene: scene_7


 91%|█████████ | 10/11 [41:47<04:20, 260.67s/it]

Saved 77758 patches for scene_7

Processing scene: scene_11


100%|██████████| 11/11 [46:24<00:00, 253.15s/it]

Saved 82916 patches for scene_11


In [33]:
import json 
import re

In [34]:
PATCH_SIZE = 128
def parse_patch_filename(filename):
    """
    Example filename:
    scene_4_p012_034.npy
    """
    match = re.search(r"_p(\d+)_([0-9]+)\.npy$", filename)
    if match is None:
        return None, None

    row = int(match.group(1))
    col = int(match.group(2))
    return row, col

In [11]:
def create_scene_metadata(scene_dir, scene_name):
    meta = []

    for fname in os.listdir(scene_dir):
        if not fname.endswith(".npy"):
            continue

        row, col = parse_patch_filename(fname)
        if row is None:
            continue

        x_offset = col * PATCH_SIZE
        y_offset = row * PATCH_SIZE

        meta.append({
            "filename": fname,
            "x_offset": x_offset,
            "y_offset": y_offset
        })

    meta_path = os.path.join(scene_dir, f"{scene_name}_patch_meta.json")

    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Metadata saved for {scene_name}: {len(meta)} patches")


## create metadata json file for all patches

In [35]:
for scene_name in os.listdir(LOCAL_PATCH_DIR):
    scene_dir = os.path.join(LOCAL_PATCH_DIR, scene_name)

    if not os.path.isdir(scene_dir):
        continue

    create_scene_metadata(scene_dir, scene_name)


Metadata saved for scene_1: 42524 patches
Metadata saved for scene_10: 106411 patches
Metadata saved for scene_11: 84789 patches
Metadata saved for scene_2: 16170 patches
Metadata saved for scene_3: 24745 patches
Metadata saved for scene_4: 84951 patches
Metadata saved for scene_5: 113948 patches
Metadata saved for scene_6: 35619 patches
Metadata saved for scene_7: 81428 patches
Metadata saved for scene_8: 93792 patches
Metadata saved for scene_9: 38809 patches


### load soil map reference

In [36]:
from utils.geo_labeling import load_soil_map
SOIL_MAP_PATH = "../Dataset/soil_classification/soil_ref/final_soil_map_wb.tif"
soil_ds = load_soil_map(SOIL_MAP_PATH)

print("Soil map loaded:")
print("CRS : ", soil_ds.crs)

Soil map loaded:
CRS :  EPSG:4326


In [40]:
import shutil

FINAL_DIR = "../Dataset/soil_classification/final_dataset"
DOMINANCE_THRESHOLD = 0.5


In [44]:
from utils.geo_labeling import label_patch_majority
import json 

os.makedirs(FINAL_DIR, exist_ok=True)

# TEMP: everything goes to train first
TRAIN_DIR = os.path.join(FINAL_DIR, "train")
os.makedirs(TRAIN_DIR, exist_ok=True)

label_counts = {}

for scene in os.listdir(LOCAL_PATCH_DIR):
    scene_dir = os.path.join(LOCAL_PATCH_DIR, scene)
    meta_path = os.path.join(scene_dir, f"{scene}_patch_meta.json")

    with open(meta_path) as f:
        metadata = json.load(f)

    for m in metadata:
        label, ratio = label_patch_majority(
            soil_ds=soil_ds,
            x_offset=m["x_offset"],
            y_offset=m["y_offset"],
            patch_size=PATCH_SIZE,
            dominance_threshold=DOMINANCE_THRESHOLD
        )

        if label is None:
            continue  # discard mixed patches

        label = str(label)

        dst_dir = os.path.join(TRAIN_DIR, label)
        os.makedirs(dst_dir, exist_ok=True)

        src_file = os.path.join(scene_dir, m["filename"])
        dst_file = os.path.join(dst_dir, m["filename"])

        shutil.copy(src_file, dst_file)

        label_counts[label] = label_counts.get(label, 0) + 1


In [45]:
print("patch count per soil class :")
for label, count in label_counts.items():
    print(f"Soil Class {label}: {count} patches")

patch count per soil class :
Soil Class 0: 1999 patches
Soil Class 18: 145 patches
Soil Class 6: 128 patches
Soil Class 11: 399 patches
